In [19]:
import pandas as pd
import numpy as np
import os
import tqdm

In [20]:
hp_id_path = r'V:\dunwei\MACE\dataset\SKNA_signal/'
hp_signal_file = r'V:\dunwei\MACE\dataset\SKNA_signal\conversion_data\patient/' # change to healthy or patient
save_path = r'V:\dunwei\MACE\dataset\pre_process'
save_file_mame = 'P_NA_feature.csv' # change to healthy or patient

In [21]:
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Created directory: {save_path}")
else:
    print(f"Directory already exists: {save_path}")

Directory already exists: V:\dunwei\MACE\dataset\pre_process


In [22]:
hp_ids = pd.read_csv(os.path.join(hp_id_path, 'MACE_p_id.csv'))# change to healthy or patient
hp_signal_ids = hp_ids['research_ID']
hp_conversion_ids = hp_ids['conversion_ID']

In [23]:
sample_rate = 10000  # sample rate is 1000 Hz

In [24]:
na_results_list = []
for i in tqdm.tqdm(range(len(hp_ids)), desc="Processing files"): # len(hp_ids)
    try:
        hp_signal_id = hp_signal_ids[i]
        # print(f"Processing index {i+1}, research_ID: {hp_signal_id}")
        signal_data = pd.read_csv(os.path.join(hp_signal_file, f'{hp_signal_id}.csv'))
        ch1_signal = signal_data.iloc[:, 1]
        if len(ch1_signal) >= 5 * 60 * sample_rate:
            ch1_signal = ch1_signal[ :(5 * 60 * sample_rate)]
            na_alpha = ch1_signal.abs().mean(axis=0)
            na_beta = (ch1_signal-ch1_signal.mean(axis=0)).abs().mean()
            y_i = (ch1_signal - ch1_signal.mean(axis=0)).abs()
            na_beta_std = y_i.std()
            na_data = {'research_ID': hp_conversion_ids[i], 'NA_alpha': na_alpha, 'NA_beta': na_beta, 'NA_beta_std': na_beta_std}
            na_results_list.append(na_data)
        else:
            print(f"Signal data for research_ID {hp_signal_ids[i]} is shorter than 5 minutes, skipping.\n")
            continue

    except Exception as e:
        print(f"Error processing research_ID {hp_signal_ids[i]}: {e}")
        continue

NA_data_df = pd.DataFrame(na_results_list)
print("\nResult:")
display(NA_data_df)

NA_data_df.to_csv(os.path.join(save_path, save_file_mame), index=False)

Processing files: 100%|██████████| 90/90 [03:18<00:00,  2.20s/it]


Result:


,research_ID,NA_alpha,NA_beta,NA_beta_std
0,MACE021,101.077016,87.539188,170.974756
1,MACE028,81.054911,63.697320,63.445154
2,MACE033,89.943586,77.839982,381.078063
3,MACE044,48.628487,36.874053,65.756009
4,MACE046,89.940086,72.618473,73.600663
...,...,...,...,...
85,MACE503,86.450658,67.785089,83.102438
86,MACE505,164.457202,161.575621,219.894200
87,MACE507,740.840686,1225.360769,1814.354510
88,MACE508,702.346337,705.890536,1334.168832
